# 00 — Environment check (CPU FIRST)

KP-14 / KP-15. Single Kaggle identity. Do not request a GPU until this cell writes `env_check.json` with `gpu_needed=false` or a later GPU session.
Do not import transformers here. Tokens stay in **Kaggle Secrets**.

In [ ]:
import json, os, platform, shutil, sys
info = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "disk": {},
}
for path in ("/kaggle/working", "/tmp", "/kaggle/temp", "/dev/shm", "/"):
    if os.path.exists(path):
        u = shutil.disk_usage(path)
        info["disk"][path] = {"free_gb": round(u.free/1e9, 2), "total_gb": round(u.total/1e9, 2)}
try:
    import torch
    info["torch"] = torch.__version__
    info["cuda"] = bool(torch.cuda.is_available())
    info["n_gpu"] = torch.cuda.device_count() if torch.cuda.is_available() else 0
    if info["cuda"]:
        info["gpu0"] = torch.cuda.get_device_name(0)
        info["vram0_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
except Exception as e:
    info["torch"] = None
    info["cuda"] = False
    info["n_gpu"] = 0
    info["torch_note"] = str(e)[:160]
qwen = "/kaggle/input/council-weights/Qwen__Qwen2.5-0.5B-Instruct/model.safetensors"
alts = [
    qwen,
    "weights_offline/modelscope/Qwen__Qwen2.5-0.5B-Instruct/model.safetensors",
]
info["qwen05"] = next((p for p in alts if os.path.exists(p)), None)
info["scratch"] = "/tmp" if os.path.isdir("/tmp") else "/kaggle/temp"
info["write_outputs_to"] = "/kaggle/working"
info["honesty"] = "Usually one T4 16GB, not Dual-T4. 70B is not a plan. CPU session must not burn GPU quota."
print(json.dumps(info, indent=2))
os.makedirs("/kaggle/working", exist_ok=True)
open("/kaggle/working/env_check.json", "w").write(json.dumps(info, indent=2))